# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [4]:
df = pd.read_csv(r"D:\Internship\Week1\Task1-Week1\data\raw\content_refresh_anonymized.csv")

df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [5]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [6]:
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

In [7]:
print(df["is_declining_label"].value_counts())

is_declining_label
1    16262
0    13738
Name: count, dtype: int64


In [31]:
X = df.drop(columns=[
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "is_declining_label"
])

# Convert categorical columns into dummy variables
X = pd.get_dummies(X, drop_first=True)

# Target
y = df["is_declining_label"]

### train_test

In [32]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

### RandomForestClassifier

In [33]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)

RandomForestClassifier(n_estimators=200, random_state=42)

In [34]:
y_pred = rf.predict(X_test)

In [35]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.7013333333333334

Classification Report
              precision    recall  f1-score   support

           0       0.69      0.63      0.66      2748
           1       0.71      0.76      0.73      3252

    accuracy                           0.70      6000
   macro avg       0.70      0.70      0.70      6000
weighted avg       0.70      0.70      0.70      6000


Confusion Matrix
[[1724 1024]
 [ 768 2484]]


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


For this project, I selected the **Random Forest Classifier** because the task is a binary classification problem that predicts whether content is declining or not.

Random Forest works well with tabular datasets containing both numerical and categorical features. It combines multiple decision trees to improve prediction accuracy and reduce overfitting. Another advantage is that it provides feature importance, making the model easier to interpret.

Based on the evaluation results, Random Forest produced reliable performance and was selected as the final model.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*



The dataset was divided into **80% training data** and **20% testing data** using `train_test_split()` with `random_state = 42`.

Stratified sampling (`stratify = y`) was used to maintain the same class distribution in both the training and testing datasets.

This split provides an honest evaluation because the model is tested on unseen data and the same split can be reused for comparison with the Week 4 baseline.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*



The Random Forest model was trained using the training dataset and evaluated on the test dataset using the same train-test split.

The Week 4 baseline was a rule-based ranking approach that prioritized stale content with high impressions. It produced a ranked action queue rather than a classification accuracy score.

The Random Forest model was evaluated using classification metrics and achieved the following result:

| Model | Evaluation |
|--------|------------|
| Week 4 Baseline | Rule-based ranking (no accuracy reported) |
| Random Forest | **70.13% Accuracy** |

The Random Forest model achieved an accuracy of **70.13%** on unseen test data. This provides an honest evaluation using historical features after removing label-derived information.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*



The Random Forest model achieved an accuracy of **70.13%** on the test dataset. Although the model performed reasonably well, it still made some incorrect predictions.

### Error Analysis

From the confusion matrix:

- **False Positives (1024):** Some pages were predicted as declining even though they were actually stable.
- **False Negatives (768):** Some declining pages were predicted as stable.

The model identified declining content better than stable content, but prediction errors still exist.

### Classification Performance

- Precision (Class 1): **0.71**
- Recall (Class 1): **0.76**
- F1-score (Class 1): **0.73**

This indicates that the model performs reasonably well in identifying declining pages while maintaining balanced precision and recall.

### Feature Interpretation

The model mainly relied on features such as:

- impressions_90d
- avg_position
- days_with_impressions
- content_age_days
- ctr

These features describe page visibility, search performance, content age, and engagement, making them useful for predicting declining content.

### Overall Interpretation

The Random Forest model learned meaningful patterns from the historical data and generalized reasonably well on unseen test data. However, some prediction errors remain because content performance can also be influenced by factors such as seasonality, changing search intent, and external events that are not included in the dataset.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.